In [26]:
import datetime
import netCDF4 as nc
import numpy as np
import pandas as pd

### Load data

In [ ]:
filepath = 'ME4OH_EN411_OFAM3/data/en4.1.1/1993-2014/ofam3-jra55.all.EN.4.1.1.f.profiles.g10.199301.update.extra.anom.199301_201412.nc'
file2read = nc.MFDataset(filepath,'r')
print(file2read.variables)

{'en4_lat': <class 'netCDF4.Variable'>
float32 en4_lat(number_of_profile)
    units: degree North
unlimited dimensions: 
current shape = (10033,)
filling on, default _FillValue of 9.969209968386869e+36 used, 'en4_lon': <class 'netCDF4.Variable'>
float32 en4_lon(number_of_profile)
    units: degree East
unlimited dimensions: 
current shape = (10033,)
filling on, default _FillValue of 9.969209968386869e+36 used, 'en4_ymd': <class 'netCDF4.Variable'>
float32 en4_ymd(number_of_profile, ymd)
    units: [year month day-of-month]
unlimited dimensions: 
current shape = (10033, 3)
filling on, default _FillValue of 9.969209968386869e+36 used, 'en4_ydec': <class 'netCDF4.Variable'>
float32 en4_ydec(number_of_profile)
    units: decimal year
unlimited dimensions: 
current shape = (10033,)
filling on, default _FillValue of 9.969209968386869e+36 used, 'en4_maxdepth': <class 'netCDF4.Variable'>
float32 en4_maxdepth(number_of_profile)
    units: m
unlimited dimensions: 
current shape = (10033,)
fillin

In [ ]:
lat = file2read["ts_lat"][:].data  # Latitude
long = file2read["ts_lon"][:].data  # Longitude
time = file2read["en4_ymd"][:].data  # Year , month , day
mask = file2read["dohc_mask_by_en4_maxdepth"][:].data # Valid profile mask
dohc1 = file2read["dohc"][:, 0].data  # Layer 1 OHC anomaly target (NOTE: index 0 in python, index 1 in R)
salt = file2read["salt"][:].data  # Salinity

### Filter data
To valid Layer-1 profiles according to mask and remove NaNs

In [ ]:
idx = [i for i in range(len(mask)) if mask[i, 0] == 1]
filtered_dohc1 = np.array([dohc1[i] for i in idx])
filtered_lat = np.array([lat[i] for i in idx])
filtered_long = np.array([long[i] for i in idx])
filtered_date = np.array([datetime.date(int(time[i, 0]), int(time[i, 1]), int(time[i, 2])) for i in idx])

In [30]:
ohc_df = pd.DataFrame({
    "Date": filtered_date,
    "Latitude": filtered_lat,
    "Longitude": filtered_long,
    "OHC": filtered_dohc1
})
ohc_df = ohc_df.dropna()
ohc_df

,Date,Latitude,Longitude,OHC
0,1993-01-18,-69.250000,271.950012,0.344897
1,1993-01-27,-68.949997,290.149994,0.344191
2,1993-01-27,-68.949997,290.450012,0.344145
3,1993-01-14,-68.949997,301.049988,0.342160
4,1993-01-15,-68.949997,302.049988,0.342066
...,...,...,...,...
6782,1993-01-25,73.849998,31.250000,0.347506
6783,1993-01-18,74.050003,23.049999,0.347372
6784,1993-01-19,74.050003,25.049999,0.348112
6785,1993-01-24,74.050003,27.049999,0.347161


# EDA

- Amount of data points across time
- Amount of data across space
- Plot OHC vs lat long for a single month?